# Exploring Chunking Strategies

In this notebook, we are going to explore different chunking strategies.

Chunking is the process of splitting a document into smaller pieces (chunks). The idea is that each chunk can be individually indexed, embedded and retrieved. Using a strong chunking strategy makes Retrieval Augmented Generation systems better.

Requirements, let's get the necessary packages installed

In [ ]:
from aup_config import aup_setup
aup_setup()

In [ ]:
import os
import requests
import re

import langchain_core
from langchain_text_splitters import TokenTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS

In [ ]:
base_url = 'https://docs.amd.com/api/khub/maps/wsGrDyp6~9qclJFHVNa2XQ/attachments/U2wv_UklnZP1~CevZS_N4Q-wsGrDyp6~9qclJFHVNa2XQ/content?download=true&Ft-Calling-App=ft%2Fturnkey-portal&Ft-Calling-App-Version=5.1.22'
download_dir = 'data_hls'
pdf_filename = 'vitis_hls_ug.pdf'

os.makedirs(download_dir, exist_ok=True)
if not os.path.isfile(os.path.join(download_dir, pdf_filename)):
    response = requests.get(base_url, stream=True)
    if response.status_code == 200:
        pdf_path = os.path.join(download_dir, pdf_filename)
        with open(pdf_path, 'wb') as file: # this triggers some activity in the NPU
            file.write(response.content)

In [ ]:
loader = PyPDFLoader(os.path.join(download_dir, pdf_filename), mode="page")

You will see that we get a single document, this is because the mode is set to `single`. If the mode is set to `page` (default), we will get one document per PDF page. However, as we want to explore chunking strategies, a single document is more suitable.

In [ ]:
pdf_doc = loader.load()

In [ ]:
docs = pdf_doc[23:38].copy()
len(docs)

cleanup

In [ ]:
for d in docs:
    d.page_content = d.page_content.replace('\nSend Feedback', '')
    d.page_content = re.sub(r'^Vitis HLS User Guide\s*.*$', '',d.page_content, flags=re.MULTILINE)
    d.page_content = re.sub(r'^UG1399\s*.*$', '',d.page_content, flags=re.MULTILINE)

In [ ]:
text = ""
for d in docs:
    text += d.page_content

print(f'{len(text)} character in the text')

In [ ]:
docs_merged = [langchain_core.documents.base.Document(text)]

## Chunking Strategies

This covers the implementation of some chunking strategies. Let's define chunk_size and chunk_overlap globally.

In [ ]:
chunk_size=1024
chunk_overlap=128

First, let's define a function to return statistics based on the documents we will get from the splitters.

In [ ]:
def chunking_stats(chunks: list[langchain_core.documents.base.Document]):
    total_documents = len(chunks)
    if total_documents < 1:
        return 0, 0, 0
    total_length = sum(len(chunk.page_content) for chunk in chunks)
    avg_length = total_length / total_documents
    return total_documents, total_length, avg_length

### Simple Chunking

In [ ]:
text_splitter_fixed = TokenTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
documents_fixed_split = text_splitter_fixed.split_documents(docs_merged)
res = chunking_stats(documents_fixed_split)
print(f"Number of chunks {res[0]}, total characters {res[1]:,}, average characters {res[2]:.2f}")

### Recursive chunking

In [ ]:
text_splitter_recursive = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
documents_recursive_split = text_splitter_recursive.split_documents(docs_merged)
res = chunking_stats(documents_recursive_split)
print(f"Number of chunks {res[0]}, total characters {res[1]:,}, average characters {res[2]:.2f}")

### Semantic Chunking

In [ ]:
embed_model = OllamaEmbeddings(model="nomic-embed-text:v1.5")
semantic_splitter = SemanticChunker(embed_model, breakpoint_threshold_type="percentile")

In [ ]:
document_semantic_split = semantic_splitter.split_documents(docs_merged)
res = chunking_stats(document_semantic_split)
print(f"Number of chunks {res[0]}, total characters {res[1]:,}, average characters {res[2]:.2f}")

### LLM chunking

LLM chunking uses Large Language Models to extract self-contained concepts or propositions. This gives the highest semantic coherence and is very useful for reasoning tasks, but it is computationally expensive during indexing.

https://community.databricks.com/t5/technical-blog/the-ultimate-guide-to-chunking-strategies-for-rag-applications/ba-p/113089#3273

In [ ]:
llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0,
)

In [ ]:
chunking_prompt = ChatPromptTemplate.from_template("""
    You are an expert processing technical documents. Your task is to split the following document into
    meaningful chunks. Follow these guidelines:

    1. Chunks should contain complete ideas or concepts, each chunk should be understandable on its own
    2. More complex sections should be in smaller chunks
    3. Chunks should not be smaller than 15 words
    4. Remove headers that bring no value to the understanding of the content, such as "Chapter 1", "Section 2.3", etc
    5. Keep related information together
    6. Do not split code snippets or tables if possible
    7. Remove all `\n`

    DOCUMENT:
    {document}

    Return ONLY a valid list of strings, where each string is a chunk.
    Format your response as:
    [
      "chunk1 text",
      "chunk2 text",
      ...
    ]
    Do not forget to open and close the list with square brackets, and to put each chunk between double quotes.
    Do not include any explanations or additional text outside the list.
    """
)

In [ ]:
chunking_chain = chunking_prompt | llm
llm_response = chunking_chain.invoke({"document": docs_merged[0].page_content})

In [ ]:
llm_split = eval(llm_response.content)

In [ ]:
documents_llm_split = [langchain_core.documents.base.Document(chunk) for chunk in llm_split]
res = chunking_stats(documents_llm_split)
print(f"Number of chunks {res[0]}, total characters {res[1]:,}, average characters {res[2]:.2f}")

## Data Base Similarity Check

Now, we are going to create vector databases using the documents created with each chunking technique that we explore above.

In [ ]:
vectorstoredb_fixed = FAISS.from_documents(documents_fixed_split, embed_model)
vectorstoredb_recursive = FAISS.from_documents(documents_recursive_split, embed_model)
vectorstoredb_semantic = FAISS.from_documents(document_semantic_split, embed_model)
vectorstoredb_llm = FAISS.from_documents(documents_llm_split, embed_model)

Now let's query the four vector databases, each of them has ingested content split with different chunking techniques.

In [ ]:
query="What is one of the most important constructs in your program?"

In [ ]:
similarity_result_fixed = vectorstoredb_fixed.similarity_search(query)
similarity_result_fixed[0].page_content

In [ ]:
similarity_result_recursive = vectorstoredb_recursive.similarity_search(query)
similarity_result_recursive[0].page_content

In [ ]:
similarity_result_semantic = vectorstoredb_semantic.similarity_search(query)
similarity_result_semantic[0].page_content

In [ ]:
similarity_result_llm = vectorstoredb_llm.similarity_search(query)
similarity_result_llm[0].page_content

The number of *similar* answers provided by the vector database is given by the argument `k` when you call `similarity_search(query)`, the default is 4. Let's modify this value and check the answers.

In [ ]:
similarity_result_llm = vectorstoredb_llm.similarity_search(query, k=10)
for idx, sim in enumerate(similarity_result_llm):
    print(f'{idx=}: {sim.page_content}\n')

Let's run another query

In [ ]:
query = "What is a stream"

In [ ]:
similarity_result_fixed = vectorstoredb_fixed.similarity_search(query)
similarity_result_fixed[0].page_content

In [ ]:
similarity_result_recursive = vectorstoredb_recursive.similarity_search(query)
similarity_result_recursive[0].page_content

In [ ]:
similarity_result_semantic = vectorstoredb_semantic.similarity_search(query)
similarity_result_semantic[0].page_content

In [ ]:
similarity_result_llm = vectorstoredb_llm.similarity_search(query)
similarity_result_llm[0].page_content

## Exercise for the Reader

1. Change `chunk_size` and `chunking_overlap`
1. Try with a different VectorDB such as [Chroma](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma), it should be as easy as replacing `FAISS` with `Chroma`

## Conclusion

This notebook showcased a few chunking techniques used in RAG systems, we explore how to create documents by using said chunking techniques. We also ingested these documents into vector databases and ran similarity search to check visually quality of results.

----------

Content curated by the AMD University Program team.

Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved.

SPDX-License-Identifier: MIT